# CSCI 485 Final Project — Group 3
## What Makes a Startup Succeed?
### Predicting Acquisition Outcomes Using the Global Startup Success Dataset

**Team Members:** Andy Chen · Devon Rhodes · Muhammad Hasham Hussain · Prabin Subedi


---
## Stage 1 — Data Loading & Preprocessing
*Responsible: Andy Chen*

Load the dataset, inspect structure, handle encoding, remove duplicates, and prepare the feature matrix.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
RANDOM_STATE = 42


In [ ]:
# Load dataset
df = pd.read_csv("global_startup_success_dataset.csv")
print(f"Shape: {df.shape}")
print(f"\nColumn dtypes:")
print(df.dtypes)


In [ ]:
# Check data quality
print(f"Missing values:\n{df.isnull().sum()}")
print(f"\nDuplicates: {df.duplicated().sum()}")
df.drop_duplicates(inplace=True)
print(f"\nShape after dedup: {df.shape}")


In [ ]:
# Preview
df.head()


In [ ]:
# Encode binary targets
df["Acquired"] = (df["Acquired?"] == "Yes").astype(int)
df["IPO"]      = (df["IPO?"]      == "Yes").astype(int)

# Encode categorical features
le = LabelEncoder()
df["Industry_enc"]     = le.fit_transform(df["Industry"])
df["Country_enc"]      = le.fit_transform(df["Country"])
df["FundingStage_enc"] = le.fit_transform(df["Funding Stage"])

print("Acquisition distribution:")
print(df["Acquired?"].value_counts())
print(f"\nClass balance: {df['Acquired'].mean():.2%} acquired")


In [ ]:
# Feature selection & train/test split
FEATURES = [
    "Total Funding ($M)",
    "Number of Employees",
    "Annual Revenue ($M)",
    "Valuation ($B)",
    "Success Score",
    "Customer Base (Millions)",
    "Social Media Followers",
    "IPO",
    "Industry_enc",
    "Country_enc",
    "FundingStage_enc",
]

X = df[FEATURES]
y = df["Acquired"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")


---
## Stage 2 — Exploratory Data Analysis (EDA)
*Responsible: Devon Rhodes*

Visualize acquisition patterns across industry, country, funding stage, and numeric correlations.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("EDA: Acquisition Patterns", fontsize=15, fontweight="bold")

# 2.1 Acquisition rate by Industry
acq_by_industry = (
    df.groupby("Industry")["Acquired"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
acq_by_industry.columns = ["Industry", "Acquisition Rate"]
axes[0, 0].bar(acq_by_industry["Industry"], acq_by_industry["Acquisition Rate"],
               color="steelblue", edgecolor="white")
axes[0, 0].set_title("Acquisition Rate by Industry")
axes[0, 0].set_xlabel("Industry")
axes[0, 0].set_ylabel("Acquisition Rate")
axes[0, 0].tick_params(axis="x", rotation=30)
axes[0, 0].set_ylim(0, 1)

# 2.2 Acquisition rate by Country (Top 10, n >= 30)
acq_by_country = (
    df.groupby("Country")["Acquired"]
    .agg(["mean", "count"])
    .query("count >= 30")
    .sort_values("mean", ascending=False)
    .head(10)
    .reset_index()
)
axes[0, 1].bar(acq_by_country["Country"], acq_by_country["mean"],
               color="coral", edgecolor="white")
axes[0, 1].set_title("Acquisition Rate by Country (Top 10, n≥30)")
axes[0, 1].set_xlabel("Country")
axes[0, 1].set_ylabel("Acquisition Rate")
axes[0, 1].tick_params(axis="x", rotation=45)
axes[0, 1].set_ylim(0, 1)

# 2.3 Acquisition rate by Funding Stage
stage_order = ["Seed", "Series A", "Series B", "Series C", "Series D", "IPO"]
acq_by_stage = (
    df[df["Funding Stage"].isin(stage_order)]
    .groupby("Funding Stage")["Acquired"]
    .mean()
    .reindex(stage_order)
    .reset_index()
)
acq_by_stage.columns = ["Funding Stage", "Acquisition Rate"]
axes[1, 0].bar(acq_by_stage["Funding Stage"], acq_by_stage["Acquisition Rate"],
               color="mediumseagreen", edgecolor="white")
axes[1, 0].set_title("Acquisition Rate by Funding Stage")
axes[1, 0].set_xlabel("Funding Stage")
axes[1, 0].set_ylabel("Acquisition Rate")
axes[1, 0].set_ylim(0, 1)

# 2.4 Correlation Heatmap
num_cols = [
    "Total Funding ($M)", "Number of Employees", "Annual Revenue ($M)",
    "Valuation ($B)", "Success Score", "Customer Base (Millions)",
    "Social Media Followers", "Acquired",
]
corr_matrix = df[num_cols].corr()
sns.heatmap(corr_matrix, ax=axes[1, 1], annot=True, fmt=".2f",
            cmap="coolwarm", center=0, linewidths=0.5, annot_kws={"size": 7})
axes[1, 1].set_title("Correlation Heatmap")
axes[1, 1].tick_params(axis="x", rotation=45, labelsize=7)
axes[1, 1].tick_params(axis="y", labelsize=7)

plt.tight_layout()
plt.savefig("eda_plots.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Summary tables
print("Acquisition Rate by Industry:")
print(acq_by_industry.to_string(index=False))
print("\nAcquisition Rate by Funding Stage:")
print(acq_by_stage.to_string(index=False))


---
## Stage 3 — Modeling & Evaluation
*Responsible: Muhammad Hasham Hussain*

Train Decision Tree (baseline), tuned Decision Tree, and Random Forest. Compare using Accuracy, Precision, Recall, and F1-Score.


In [ ]:
def evaluate_model(name, y_true, y_pred):
    return {
        "Model":     name,
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall":    recall_score(y_true, y_pred),
        "F1-Score":  f1_score(y_true, y_pred),
    }


In [ ]:
# 3.1 Decision Tree — Baseline
dt = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
dt_metrics = evaluate_model("Decision Tree (depth=5)", y_test, y_pred_dt)
print("Decision Tree (baseline):")
print(classification_report(y_test, y_pred_dt, target_names=["Not Acquired", "Acquired"]))


In [ ]:
# 3.2 Decision Tree — Tuned
dt_tuned = DecisionTreeClassifier(
    max_depth=8, min_samples_split=20, min_samples_leaf=10, random_state=RANDOM_STATE
)
dt_tuned.fit(X_train, y_train)
y_pred_dt_tuned = dt_tuned.predict(X_test)
dt_tuned_metrics = evaluate_model("Decision Tree (tuned)", y_test, y_pred_dt_tuned)
print("Decision Tree (tuned):")
print(classification_report(y_test, y_pred_dt_tuned, target_names=["Not Acquired", "Acquired"]))


In [ ]:
# 3.3 Random Forest
rf = RandomForestClassifier(
    n_estimators=100, max_depth=10, min_samples_split=10,
    min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
rf_metrics = evaluate_model("Random Forest", y_test, y_pred_rf)
print("Random Forest:")
print(classification_report(y_test, y_pred_rf, target_names=["Not Acquired", "Acquired"]))


In [ ]:
# 3.4 Model Comparison Table (Vertical)
results_df = pd.DataFrame([dt_metrics, dt_tuned_metrics, rf_metrics]).set_index("Model").round(4)
print("Model Comparison (Vertical — our own iterations):")
results_df


In [ ]:
# 3.5 Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Confusion Matrices", fontsize=13, fontweight="bold")

for ax, (name, y_pred) in zip(
    axes,
    [("DT (depth=5)", y_pred_dt), ("DT (tuned)", y_pred_dt_tuned), ("Random Forest", y_pred_rf)],
):
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["Not Acquired", "Acquired"]).plot(
        ax=ax, colorbar=False, cmap="Blues"
    )
    ax.set_title(name)

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# 3.6 Feature Importance — Random Forest
feat_importance = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)

plt.figure(figsize=(8, 6))
colors = ["coral" if v == feat_importance.max() else "steelblue" for v in feat_importance]
plt.barh(feat_importance.index, feat_importance.values, color=colors, edgecolor="white")
plt.title("Random Forest: Feature Importance", fontsize=13, fontweight="bold")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

print("Feature importance ranking:")
print(feat_importance.sort_values(ascending=False).round(4).to_string())


In [ ]:
# 3.7 Decision Tree Visualization (for code walkthrough)
plt.figure(figsize=(20, 8))
plot_tree(
    dt, feature_names=FEATURES, class_names=["Not Acquired", "Acquired"],
    filled=True, rounded=True, fontsize=8, max_depth=3,
)
plt.title("Decision Tree Structure (first 3 levels)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("decision_tree_viz.png", dpi=120, bbox_inches="tight")
plt.show()


---
## Stage 4 — Results Analysis & Conclusion
*Responsible: Prabin Subedi*


In [ ]:
# Horizontal comparison vs majority-class baseline
majority_acc = max(y_test.mean(), 1 - y_test.mean())
print(f"Majority class baseline accuracy : {majority_acc:.4f}")
print(f"Decision Tree improvement        : {dt_metrics['Accuracy']  - majority_acc:+.4f}")
print(f"Random Forest improvement        : {rf_metrics['Accuracy']  - majority_acc:+.4f}")


### Summary

**Best model:** Random Forest

**Top 3 features by importance:**
1. Number of Employees
2. Annual Revenue ($M)
3. Valuation ($B)

**What worked:**
- Random Forest outperformed Decision Tree on all metrics
- Feature importance provides direct interpretability
- Balanced dataset (≈50/50) avoids class imbalance problems

**What did not work:**
- Decision Tree (depth=5) underfits — accuracy near random baseline
- Tech Stack dropped due to high cardinality (not encoded)
- Low overall accuracy (~50%) suggests weak signal in the dataset features for predicting acquisition

**Future work:**
- Try XGBoost or LightGBM
- One-hot encode Tech Stack
- Apply GridSearchCV for hyperparameter tuning
- Use SHAP values for deeper feature interpretability
